In [17]:
from openai import OpenAI
from dotenv import load_dotenv
import mlflow
import os

load_dotenv()

DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_ENDPOINT = os.environ.get("DATABRICKS_ENDPOINT")

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{DATABRICKS_HOST}/serving-endpoints",
)

# Specify the tracking server URI, e.g. http://localhost:5000
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("openai-api-basic-chat-observability")
print("MLflow experiment set: openai-api-basic-chat-observability")

MLflow experiment set: openai-api-basic-chat-observability


In [ ]:
@mlflow.trace(name="chat_completion", span_type="LLM")
def _traced_chat_completion(messages, temperature):
    return client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=messages,
        temperature=temperature,
    )


def chat_once(user_message, history=None, temperature=0.2):
    """Send one chat request and log observability data to MLflow runs + traces."""
    history = history or []
    messages = history + [{"role": "user", "content": user_message}]

    with mlflow.start_run(run_name="openai_chat_turn") as run:
        response = _traced_chat_completion(messages=messages, temperature=temperature)

        assistant_text = response.choices[0].message.content or ""
        usage = response.usage

        mlflow.log_param("model", DATABRICKS_ENDPOINT)
        mlflow.log_param("temperature", temperature)
        mlflow.log_param("input_chars", len(user_message))
        mlflow.log_metric("prompt_tokens", usage.prompt_tokens if usage else 0)
        mlflow.log_metric("completion_tokens", usage.completion_tokens if usage else 0)
        mlflow.log_metric("total_tokens", usage.total_tokens if usage else 0)

        mlflow.log_text(user_message, "input_prompt.txt")
        mlflow.log_text(assistant_text, "assistant_response.txt")
        mlflow.set_tag("component", "chat")
        mlflow.set_tag("status", "success")

        print(f"MLflow run logged: {run.info.run_id}")
        print(f"MLflow trace logged: {mlflow.get_last_active_trace_id()}")

    updated_history = messages + [{"role": "assistant", "content": assistant_text}]
    return assistant_text, updated_history


In [ ]:
chat_history = []
assistant_reply, chat_history = chat_once(
    "Give me a two-line intro to MLflow observability.",
    history=chat_history,
    temperature=0.2,
 )
print("Assistant:", assistant_reply)

exp = mlflow.get_experiment_by_name("openai-api-basic-chat-observability")
if exp:
    traces = mlflow.search_traces(
        experiment_ids=[exp.experiment_id],
        max_results=5,
    )
    print("Recent traces:")
    display(traces)
else:
    print("Experiment not found.")

MLflow run logged: 3e09f68028424239958c29b995826b4f
🏃 View run openai_chat_turn at: http://127.0.0.1:5000/#/experiments/1/runs/3e09f68028424239958c29b995826b4f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Assistant: Here is a two-line intro to MLflow observability:

MLflow observability enables data scientists and ML engineers to monitor and understand the performance of their machine learning models in production, tracking metrics, predictions, and data drift over time. By integrating with MLflow's model registry and tracking capabilities, observability tools provide a comprehensive view of model behavior, facilitating proactive maintenance and improvement of ML systems.


In [23]:
assistant_reply, chat_history = chat_once("Now summarize that in one sentence.", history=chat_history)
print("Assistant:", assistant_reply)

MLflow run logged: 60f799ebfce44e988fb9da57e464abf0
🏃 View run openai_chat_turn at: http://127.0.0.1:5000/#/experiments/1/runs/60f799ebfce44e988fb9da57e464abf0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Assistant: MLflow provides model observability.
